In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# Stage 16 — Reference coding solutions

Three exercises. Read the lesson and workbook first. Learner functions intentionally raise `NotImplementedError` until you write them. Reference execution is not your assessment.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
from pathlib import Path
import sys
import numpy as np
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/shape_lab").is_dir())
if str(ROOT/"src") not in sys.path:sys.path.insert(0,str(ROOT/"src"))


## 16.C1 — `stable_softmax`

Compute row-normalized softmax for a nonempty finite two-dimensional score array.

**Before coding:** work a small example on paper.

**Hint:** Subtract the row maximum before exponentiating.

In [3]:
def stable_softmax(scores):
    s=np.asarray(scores,float)
    if s.ndim!=2 or not s.shape[0] or not s.shape[1] or not np.isfinite(s).all():raise ValueError("Finite nonempty matrix required.")
    e=np.exp(s-s.max(axis=1,keepdims=True))
    return e/e.sum(axis=1,keepdims=True)

In [4]:
w=stable_softmax([[1000,1001],[0,0]])
assert np.isfinite(w).all() and np.allclose(w.sum(1),1)
assert np.allclose(w[1],[.5,.5])
assert np.allclose(w,stable_softmax(np.array([[1000,1001],[0,0]])+42))
print("16.C1: checks passed")

16.C1: checks passed


**Why this works and where it stops:** Subtracting a common constant per row leaves softmax unchanged; it prevents overflow from unnecessary common offsets.

## 16.C2 — `same_frame_mask`

Return a boolean pairwise mask allowing a token to attend only to tokens with the same frame ID.

**Before coding:** work a small example on paper.

**Hint:** Compare a column of IDs against a row of IDs using broadcasting.

In [5]:
def same_frame_mask(frame_ids):
    ids=np.asarray(frame_ids)
    if ids.ndim!=1:raise ValueError("One-dimensional IDs required.")
    return ids[:,None]==ids[None,:]

In [6]:
m=same_frame_mask([0,0,1])
assert np.array_equal(m,[[True,True,False],[True,True,False],[False,False,True]])
assert np.array_equal(m,m.T) and np.diag(m).all()
print("16.C2: checks passed")

16.C2: checks passed


**Why this works and where it stops:** The mask specifies allowed information flow. It does not learn which image points correspond.

## 16.C3 — `weighted_values`

Multiply a row-normalized attention-weight matrix by value vectors. Validate dimensions and normalization.

**Before coding:** work a small example on paper.

**Hint:** A row of weights is used to average value rows.

In [7]:
def weighted_values(weights, values):
    W=np.asarray(weights,float);V=np.asarray(values,float)
    if W.ndim!=2 or V.ndim!=2 or W.shape[1]!=V.shape[0] or not np.isfinite(W).all() or not np.isfinite(V).all() or np.any(W<0) or not np.allclose(W.sum(1),1):raise ValueError("Compatible normalized weights required.")
    return W@V

In [8]:
assert np.allclose(weighted_values([[.25,.75]],[[2,4],[6,8]]),[[5,7]])
assert np.allclose(weighted_values(np.eye(2),[[2,4],[6,8]]),[[2,4],[6,8]])
print("16.C3: checks passed")

16.C3: checks passed


**Why this works and where it stops:** The result can be calculated without a trained geometry model. It demonstrates averaging, not recovered 3D shape.